In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

#A. Préparation de données
df = pd.read_csv('/content/my_full_dataset.csv')

df['clean_text'] = df['clean_text'].fillna('').astype(str)

X = df['clean_text']
y = df['author']


# B. Encodage de la variable à prédire

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# ---------------------------------------------------------
# C. Construction des bases d'entraînement et de test
# ---------------------------------------------------------
# Split the dataset with 30% for testing and random_state=0
# We use stratify=y_encoded to handle the imbalanced dataset as requested
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.30,
    random_state=0,
    stratify=y_encoded
)

# ---------------------------------------------------------
# D. Méthodes de vectorisation
# ---------------------------------------------------------
# 1. Lexical frequency (BoW)
freq_vectorizer = CountVectorizer()
X_train_freq = freq_vectorizer.fit_transform(X_train)
X_test_freq = freq_vectorizer.transform(X_test)

# 1. One-hot encoding (Binary Count Vectorization)
# Setting binary=True acts as a One-Hot Encoder for the vocabulary presence
onehot_vectorizer = CountVectorizer(binary=True)
X_train_onehot = onehot_vectorizer.fit_transform(X_train)
X_test_onehot = onehot_vectorizer.transform(X_test)

# 2 & 3. TF-IDF Vectorization
tfidf_vectorizer = TfidfVectorizer()
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train) # Fit and transform on train
X_test_tfidf = tfidf_vectorizer.transform(X_test)       # Only transform on test

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report

# ---------------------------------------------------------
# E.1: Créer trois modèles du type MLPClassifier
# ---------------------------------------------------------
# We create a separate neural network for each type of data.
# (I added hidden_layer_sizes=(50,) and max_iter=20 so it trains fast in your TP class.
# You can remove them if your PC is fast and you have plenty of time!)
mlp_freq = MLPClassifier(hidden_layer_sizes=(50,), max_iter=20, random_state=0)
mlp_onehot = MLPClassifier(hidden_layer_sizes=(50,), max_iter=20, random_state=0)
mlp_tfidf = MLPClassifier(hidden_layer_sizes=(50,), max_iter=20, random_state=0)

# ---------------------------------------------------------
# E.2: Entrainer ces trois modèles
# ---------------------------------------------------------
# We use .fit() to train the models. Notice we ONLY use the X_train data here!
print("Entraînement des modèles en cours (cela peut prendre un moment)...")
mlp_freq.fit(X_train_freq, y_train)
mlp_onehot.fit(X_train_onehot, y_train)
mlp_tfidf.fit(X_train_tfidf, y_train)

# ---------------------------------------------------------
# E.3: Prédire les classes (Sur l'entraînement)
# ---------------------------------------------------------
# We ask the model to predict the answers for the data it just studied.
pred_train_freq = mlp_freq.predict(X_train_freq)
pred_train_onehot = mlp_onehot.predict(X_train_onehot)
pred_train_tfidf = mlp_tfidf.predict(X_train_tfidf)

# ---------------------------------------------------------
# E.4: Afficher le rapport de classification
# ---------------------------------------------------------
# We compare the true labels (y_train) with the model's predictions
print("\n=== RAPPORT : Fréquence Lexicale (BoW) ===")
print(classification_report(y_train, pred_train_freq, target_names=label_encoder.classes_))

print("\n=== RAPPORT : One-Hot Encoding ===")
print(classification_report(y_train, pred_train_onehot, target_names=label_encoder.classes_))

print("\n=== RAPPORT : TF-IDF ===")
print(classification_report(y_train, pred_train_tfidf, target_names=label_encoder.classes_))

Entraînement des modèles en cours (cela peut prendre un moment)...


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(



=== RAPPORT : Fréquence Lexicale (BoW) ===
              precision    recall  f1-score   support

         EAP       1.00      1.00      1.00      5530
         HPL       1.00      1.00      1.00      3944
         MWS       1.00      1.00      1.00      4231

    accuracy                           1.00     13705
   macro avg       1.00      1.00      1.00     13705
weighted avg       1.00      1.00      1.00     13705


=== RAPPORT : One-Hot Encoding ===
              precision    recall  f1-score   support

         EAP       1.00      1.00      1.00      5530
         HPL       1.00      1.00      1.00      3944
         MWS       1.00      1.00      1.00      4231

    accuracy                           1.00     13705
   macro avg       1.00      1.00      1.00     13705
weighted avg       1.00      1.00      1.00     13705


=== RAPPORT : TF-IDF ===
              precision    recall  f1-score   support

         EAP       1.00      1.00      1.00      5530
         HPL       1.00

/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(


In [ ]:
import time
from sklearn.metrics import classification_report

# ---------------------------------------------------------
# F.1: Prédire les classes sur les représentations de TEST
# ---------------------------------------------------------
# On utilise .predict() sur les données X_test que le modèle n'a jamais vues
pred_test_freq = mlp_freq.predict(X_test_freq)
pred_test_onehot = mlp_onehot.predict(X_test_onehot)
pred_test_tfidf = mlp_tfidf.predict(X_test_tfidf)

# ---------------------------------------------------------
# F.2: Afficher le rapport de classification
# ---------------------------------------------------------
# On compare les vraies réponses (y_test) avec les prédictions du modèle
print("\n=== RÉSULTATS DE TEST : Fréquence Lexicale (BoW) ===")
print(classification_report(y_test, pred_test_freq, target_names=label_encoder.classes_))

print("\n=== RÉSULTATS DE TEST : One-Hot Encoding ===")
print(classification_report(y_test, pred_test_onehot, target_names=label_encoder.classes_))

print("\n=== RÉSULTATS DE TEST : TF-IDF ===")
print(classification_report(y_test, pred_test_tfidf, target_names=label_encoder.classes_))


=== RÉSULTATS DE TEST : Fréquence Lexicale (BoW) ===
              precision    recall  f1-score   support

         EAP       0.77      0.82      0.79      2370
         HPL       0.83      0.75      0.79      1691
         MWS       0.77      0.78      0.78      1813

    accuracy                           0.79      5874
   macro avg       0.79      0.78      0.79      5874
weighted avg       0.79      0.79      0.79      5874


=== RÉSULTATS DE TEST : One-Hot Encoding ===
              precision    recall  f1-score   support

         EAP       0.77      0.82      0.80      2370
         HPL       0.84      0.75      0.79      1691
         MWS       0.78      0.78      0.78      1813

    accuracy                           0.79      5874
   macro avg       0.80      0.79      0.79      5874
weighted avg       0.79      0.79      0.79      5874


=== RÉSULTATS DE TEST : TF-IDF ===
              precision    recall  f1-score   support

         EAP       0.79      0.83      0.81    

In [ ]:
!pip install gensim
import numpy as np
from gensim.models import Word2Vec, FastText
import gensim.downloader as api

# ---------------------------------------------------------
# G. Préparation : Tokenisation des phrases
# ---------------------------------------------------------
# On découpe chaque texte en liste de mots (tokens)
X_train_tokens = [text.split() for text in X_train]
X_test_tokens = [text.split() for text in X_test]

# ---------------------------------------------------------
# G.1 Entraînement des modèles (Word2Vec et FastText)
# ---------------------------------------------------------
print("1. Entraînement de Word2Vec (CBOW)...")
# sg=0 signifie CBOW (Continuous Bag of Words)
w2v_cbow = Word2Vec(sentences=X_train_tokens, vector_size=100, window=5, min_count=1, sg=0)

print("2. Entraînement de Word2Vec (Skip-gram)...")
# sg=1 signifie Skip-gram
w2v_skipgram = Word2Vec(sentences=X_train_tokens, vector_size=100, window=5, min_count=1, sg=1)

print("3. Entraînement de FastText...")
fasttext_model = FastText(sentences=X_train_tokens, vector_size=100, window=5, min_count=1)

print("4. Chargement de GloVe (téléchargement la 1ère fois)...")
# On télécharge un tout petit modèle GloVe de Twitter pour que ça soit rapide (25 dimensions)
glove_wv = api.load("glove-twitter-25")

# ---------------------------------------------------------
# Fonction pour transformer une phrase en UN SEUL vecteur
# ---------------------------------------------------------
def get_sentence_vector(model_wv, tokens, vector_size):
    # On garde seulement les mots que le modèle connaît
    valid_words = [word for word in tokens if word in model_wv]

    if len(valid_words) == 0:
        return np.zeros(vector_size) # Vecteur vide si aucun mot n'est connu

    # On calcule la moyenne des vecteurs de tous les mots de la phrase
    return np.mean(model_wv[valid_words], axis=0)

# ---------------------------------------------------------
# G.2 Création des matrices finales pour le MLP
# ---------------------------------------------------------
print("\nCréation des matrices vectorielles pour l'entraînement et le test...")

# a. CBOW (taille 100)
X_train_cbow = np.array([get_sentence_vector(w2v_cbow.wv, tokens, 100) for tokens in X_train_tokens])
X_test_cbow = np.array([get_sentence_vector(w2v_cbow.wv, tokens, 100) for tokens in X_test_tokens])

# a. Skip-Gram (taille 100)
X_train_skipgram = np.array([get_sentence_vector(w2v_skipgram.wv, tokens, 100) for tokens in X_train_tokens])
X_test_skipgram = np.array([get_sentence_vector(w2v_skipgram.wv, tokens, 100) for tokens in X_test_tokens])

# c. FastText (taille 100)
X_train_fasttext = np.array([get_sentence_vector(fasttext_model.wv, tokens, 100) for tokens in X_train_tokens])
X_test_fasttext = np.array([get_sentence_vector(fasttext_model.wv, tokens, 100) for tokens in X_test_tokens])

# b. GloVe (Attention, on a téléchargé un modèle de taille 25)
X_train_glove = np.array([get_sentence_vector(glove_wv, tokens, 25) for tokens in X_train_tokens])
X_test_glove = np.array([get_sentence_vector(glove_wv, tokens, 25) for tokens in X_test_tokens])

print("Vectorisation terminée avec succès !")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 76.7 MB/s eta 0:00:00
1. Entraînement de Word2Vec (CBOW)...
2. Entraînement de Word2Vec (Skip-gram)...
3. Entraînement de FastText...
4. Chargement de GloVe (téléchargement la 1ère fois)...
[==================================================] 100.0% 104.8/104.8MB downloaded

Création des matrices vectorielles pour l'entraînement et le test...
Vectorisation terminée avec succès !


In [ ]:
import time
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report

# ---------------------------------------------------------
# H.1 Entraînement et Test sur les Word Embeddings
# ---------------------------------------------------------

# On regroupe nos nouvelles données dans des dictionnaires
vectorizations_train_emb = {
    "Word2Vec (CBOW)": X_train_cbow,
    "Word2Vec (Skip-gram)": X_train_skipgram,
    "FastText": X_train_fasttext,
    "GloVe": X_train_glove
}

vectorizations_test_emb = {
    "Word2Vec (CBOW)": X_test_cbow,
    "Word2Vec (Skip-gram)": X_test_skipgram,
    "FastText": X_test_fasttext,
    "GloVe": X_test_glove
}

# On crée nos modèles MLP (avec les mêmes paramètres qu'avant pour une comparaison juste)
models_emb = {
    "Word2Vec (CBOW)": MLPClassifier(hidden_layer_sizes=(50,), max_iter=20, random_state=0),
    "Word2Vec (Skip-gram)": MLPClassifier(hidden_layer_sizes=(50,), max_iter=20, random_state=0),
    "FastText": MLPClassifier(hidden_layer_sizes=(50,), max_iter=20, random_state=0),
    "GloVe": MLPClassifier(hidden_layer_sizes=(50,), max_iter=20, random_state=0)
}

# La boucle magique pour tout faire d'un coup !
for name in vectorizations_train_emb.keys():
    print(f"\n{'='*55}")
    print(f"### Évaluation pour la représentation : {name} ###")
    print(f"{'='*55}")

    X_train_vec = vectorizations_train_emb[name]
    X_test_vec = vectorizations_test_emb[name]
    model = models_emb[name]

    # --- Calcul du temps d'entraînement ---
    start_time = time.time()
    model.fit(X_train_vec, y_train)
    end_time = time.time()

    print(f"-> Temps d'entraînement : {end_time - start_time:.4f} secondes\n")

    # --- Rapport sur l'ENTRAÎNEMENT ---
    y_train_pred = model.predict(X_train_vec)
    print("Rapport de classification (Base d'entraînement) :")
    print(classification_report(y_train, y_train_pred, target_names=label_encoder.classes_, zero_division=0))

    # --- Rapport sur le TEST ---
    y_test_pred = model.predict(X_test_vec)
    print("Rapport de classification (Base de test) :")
    print(classification_report(y_test, y_test_pred, target_names=label_encoder.classes_, zero_division=0))


### Évaluation pour la représentation : Word2Vec (CBOW) ###


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(


-> Temps d'entraînement : 0.9369 secondes

Rapport de classification (Base d'entraînement) :
              precision    recall  f1-score   support

         EAP       0.40      1.00      0.57      5530
         HPL       0.00      0.00      0.00      3944
         MWS       0.00      0.00      0.00      4231

    accuracy                           0.40     13705
   macro avg       0.13      0.33      0.19     13705
weighted avg       0.16      0.40      0.23     13705

Rapport de classification (Base de test) :
              precision    recall  f1-score   support

         EAP       0.40      1.00      0.57      2370
         HPL       0.00      0.00      0.00      1691
         MWS       0.00      0.00      0.00      1813

    accuracy                           0.40      5874
   macro avg       0.13      0.33      0.19      5874
weighted avg       0.16      0.40      0.23      5874


### Évaluation pour la représentation : Word2Vec (Skip-gram) ###


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(


-> Temps d'entraînement : 0.9418 secondes

Rapport de classification (Base d'entraînement) :
              precision    recall  f1-score   support

         EAP       0.55      0.72      0.62      5530
         HPL       0.58      0.39      0.46      3944
         MWS       0.60      0.55      0.57      4231

    accuracy                           0.57     13705
   macro avg       0.58      0.55      0.55     13705
weighted avg       0.57      0.57      0.56     13705

Rapport de classification (Base de test) :
              precision    recall  f1-score   support

         EAP       0.54      0.69      0.60      2370
         HPL       0.57      0.36      0.44      1691
         MWS       0.56      0.55      0.56      1813

    accuracy                           0.55      5874
   macro avg       0.56      0.53      0.53      5874
weighted avg       0.56      0.55      0.54      5874


### Évaluation pour la représentation : FastText ###


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(


-> Temps d'entraînement : 0.9123 secondes

Rapport de classification (Base d'entraînement) :
              precision    recall  f1-score   support

         EAP       0.40      1.00      0.57      5530
         HPL       1.00      0.00      0.00      3944
         MWS       0.11      0.00      0.00      4231

    accuracy                           0.40     13705
   macro avg       0.50      0.33      0.19     13705
weighted avg       0.48      0.40      0.23     13705

Rapport de classification (Base de test) :
              precision    recall  f1-score   support

         EAP       0.40      1.00      0.57      2370
         HPL       0.00      0.00      0.00      1691
         MWS       0.33      0.00      0.00      1813

    accuracy                           0.40      5874
   macro avg       0.25      0.33      0.19      5874
weighted avg       0.27      0.40      0.23      5874


### Évaluation pour la représentation : GloVe ###
-> Temps d'entraînement : 0.5538 secondes

Rapport 

/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(
